# 📈 Notebook 03 — Approches de Prévision Temporelle (Forecasting)
## Projet ThermoPath : Évaluation Critique des Modèles Classiques

---

### 🎯 Objectif

Dans les notebooks précédents, nous avons :
1. **Notebook 01 (EDA)** : Visualisé la corrélation entre chocs mécaniques et dérive thermique.
2. **Notebook 02 (Tests Statistiques)** : Prouvé mathématiquement (Mann-Whitney, p < 0.05) que la vélocité thermique post-choc est significativement différente de la normale.

Avant de concevoir notre solution finale basée sur la **Détection d'Anomalies (Isolation Forest)**, nous devons évaluer l'approche classique : le **Forecasting** (prévision temporelle) à l'aide de modèles autorégressifs de type ARIMA/SARIMAX.

Ce notebook va :
- Tester la **stationnarité** de notre série temporelle.
- Construire un modèle **SARIMAX** pour prédire la température future.
- **Démontrer les limites fondamentales** de cette approche pour notre cas d'usage industriel embarqué.

<div class="alert alert-info">
<b>📐 Philosophie :</b> En ingénierie industrielle, choisir le bon outil est aussi important que la qualité du modèle. Un modèle parfait mais inutilisable en production n'a aucune valeur.
</div>

---
## 1. 🔧 Chargement des Données & Préparation

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# IMPORTS & CONFIGURATION
# ══════════════════════════════════════════════════════════════════════
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Accès aux modules du projet
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.data_prep import load_and_resample
from src.features import inject_synthetic_faults

# Imports spécifiques à ce notebook
from statsmodels.tsa.stattools import adfuller          # Test de Dickey-Fuller Augmenté
from statsmodels.tsa.statespace.sarimax import SARIMAX  # Modèle SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Style graphique ──────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({
    'figure.figsize': (14, 5), 'axes.titlesize': 14,
    'axes.labelsize': 12, 'font.size': 11, 'figure.dpi': 120,
})
COLORS = {'temp':'#1E88E5','gforce':'#E53935','pred':'#FF6F00','real':'#43A047'}
print("Librairies importees avec succes.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CHARGEMENT, INJECTION DES CHOCS & RENOMMAGE
# ══════════════════════════════════════════════════════════════════════

df = load_and_resample(file_path="../data/raw/input_data.csv", batch_id="batch001")
df = df.iloc[:5000]                                    # Sous-ensemble pour performance
df = inject_synthetic_faults(df, num_shocks=3, seed=42)

# ── Alias pour lisibilité : temp (endogène) et g_force (exogène) ────
df['temp'] = df['thermal_shipper_temp_reading']

print(f"Dimensions du DataFrame : {df.shape}")
print(f"Plage temporelle        : {df.index.min()} → {df.index.max()}")
print(f"\nApercu des variables cles :")
df[['temp', 'g_force']].describe().round(4)

---
## 2. 📊 Test de Stationnarité — Dickey-Fuller Augmenté (ADF)

La **stationnarité** est une condition fondamentale pour les modèles ARIMA/SARIMAX. Une série est stationnaire si ses propriétés statistiques (moyenne, variance) restent constantes dans le temps.

Le test **ADF (Augmented Dickey-Fuller)** formule les hypothèses suivantes :
- **H₀** : La série possède une racine unitaire → elle est **non-stationnaire**.
- **H₁** : La série ne possède pas de racine unitaire → elle est **stationnaire**.

Si `p-value < 0.05`, on rejette H₀ et la série est considérée stationnaire.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# TEST DE DICKEY-FULLER AUGMENTÉ SUR LA VARIABLE `temp`
# ══════════════════════════════════════════════════════════════════════

result = adfuller(df['temp'].dropna(), autolag='AIC')

adf_stat    = result[0]   # Statistique ADF
p_value     = result[1]   # p-value
used_lags   = result[2]   # Nombre de lags utilisés
n_obs       = result[3]   # Nombre d'observations
crit_values = result[4]   # Valeurs critiques (1%, 5%, 10%)

print("=" * 60)
print("  TEST DE DICKEY-FULLER AUGMENTÉ (ADF)")
print("  Variable testée : temp (thermal_shipper_temp_reading)")
print("=" * 60)
print(f"  Statistique ADF  : {adf_stat:.6f}")
print(f"  p-value          : {p_value:.6e}")
print(f"  Lags utilisés    : {used_lags}")
print(f"  Observations     : {n_obs}")
print("-" * 60)
print("  Valeurs critiques :")
for key, val in crit_values.items():
    marker = " <<<" if adf_stat < val else ""
    print(f"    {key} : {val:.6f}{marker}")
print("-" * 60)

if p_value < 0.05:
    print("  >>> REJET de H0 : la série SEMBLE stationnaire (p < 0.05).")
    print("  Attention : cela ne signifie pas qu'elle est facile à modéliser.")
else:
    print("  >>> ECHEC du rejet de H0 : la série est NON-STATIONNAIRE.")
    print("  Une différenciation (d >= 1) sera nécessaire pour ARIMA.")
print("=" * 60)

### 🔎 Interprétation du Test ADF

Que la p-value soit significative ou non, un point fondamental demeure :

> **Les chocs mécaniques brutaux introduisent des ruptures structurelles** dans la série temporelle. Le test ADF, conçu pour détecter des racines unitaires dans des processus **continus et lisses**, est mal adapté à ce type de dynamique.

En effet :
- Si la série apparaît stationnaire, c'est parce que les dérives post-choc sont **noyées** dans le bruit de mesure global.
- Si elle est non-stationnaire, la différenciation (`d=1`) ne résoudra pas le problème de fond : le modèle ARIMA ne sait pas qu'un **événement exogène discret** (le choc) est la cause de la dérive.

C'est le **premier signal d'alarme** contre l'approche Forecasting classique.

---
## 3. 🤖 Modélisation SARIMAX — Prédiction de la Température

Nous allons maintenant construire un modèle **SARIMAX** :
- **Variable endogène (Y)** : `temp` — la température du caisson frigorifique.
- **Variable exogène (X)** : `g_force` — le stress mécanique mesuré par l'accéléromètre.

Le **X** de SARIMAX est précisément ce qui différencie ce modèle d'un ARIMA pur : il permet d'intégrer des variables explicatives externes.

### Paramètres du modèle
- `order=(1, 1, 1)` : (p=1, d=1, q=1)
  - **p=1** : 1 terme autorégressif (AR) — la prédiction dépend de la valeur précédente.
  - **d=1** : 1 différenciation — pour rendre la série stationnaire.
  - **q=1** : 1 terme de moyenne mobile (MA) — correction basée sur l'erreur précédente.

### Split Train / Test
- **80% Train / 20% Test**, en respectant l'ordre chronologique (pas de shuffle !).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SÉPARATION TRAIN / TEST (80/20 — ordre chronologique respecté)
# ══════════════════════════════════════════════════════════════════════

split_idx = int(len(df) * 0.8)

train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

print(f"Train : {len(train)} observations  ({train.index.min()} → {train.index.max()})")
print(f"Test  : {len(test)} observations   ({test.index.min()} → {test.index.max()})")
print(f"Ratio : {len(train)/len(df)*100:.0f}% / {len(test)/len(df)*100:.0f}%")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# ENTRAÎNEMENT DU MODÈLE SARIMAX
# ══════════════════════════════════════════════════════════════════════
# order=(p, d, q) = (1, 1, 1)
#   p = 1 : un terme autorégressif (dépendance à t-1)
#   d = 1 : une différenciation (stationnarisation)
#   q = 1 : un terme de moyenne mobile (correction par erreur passée)
# exog = g_force : variable exogène (le 'X' de SARIMAX)
# ══════════════════════════════════════════════════════════════════════

print("Entraînement du modèle SARIMAX(1,1,1) en cours...")
print("(Cela peut prendre quelques minutes selon la taille du jeu de données.)\n")

model = SARIMAX(
    endog=train['temp'],                # Variable endogène : température
    exog=train[['g_force']],            # Variable exogène  : g_force
    order=(1, 1, 1),                    # Paramètres ARIMA (p, d, q)
    enforce_stationarity=False,         # Relaxer la contrainte de stationnarité
    enforce_invertibility=False,        # Relaxer la contrainte d'inversibilité
)

results = model.fit(disp=False)         # disp=False : pas de log verbose

print("Modèle entraîné avec succès.")
print(results.summary().tables[0])

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# PRÉDICTION SUR LE JEU DE TEST
# ══════════════════════════════════════════════════════════════════════

predictions = results.forecast(
    steps=len(test),
    exog=test[['g_force']]              # On fournit les exogènes du test
)

# ── Métriques d'évaluation ───────────────────────────────────────────
mae  = mean_absolute_error(test['temp'], predictions)
rmse = np.sqrt(mean_squared_error(test['temp'], predictions))

print("=" * 60)
print("  MÉTRIQUES DE PERFORMANCE — SARIMAX(1,1,1)")
print("=" * 60)
print(f"  MAE  (Mean Absolute Error)    : {mae:.4f} °C")
print(f"  RMSE (Root Mean Squared Error) : {rmse:.4f} °C")
print("=" * 60)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# VISUALISATION : PRÉDICTION vs RÉALITÉ SUR LE SET DE TEST
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# ── Subplot 1 : Température réelle vs prédite ────────────────────────
axes[0].plot(test.index, test['temp'], label='Réalité (temp)',
             color=COLORS['real'], linewidth=1, alpha=0.8)
axes[0].plot(test.index, predictions, label='Prédiction SARIMAX(1,1,1)',
             color=COLORS['pred'], linewidth=1.2, linestyle='--')
axes[0].set_ylabel('Température (°C)')
axes[0].set_title('SARIMAX(1,1,1) — Prédiction vs Réalité sur le Set de Test',
                  fontweight='bold', fontsize=14)
axes[0].legend(loc='upper right')
axes[0].fill_between(test.index, test['temp'], predictions,
                     alpha=0.15, color='red', label='Erreur')

# ── Subplot 2 : G-Force (exogène) ────────────────────────────────────
axes[1].plot(test.index, test['g_force'], label='G-Force (exogène)',
             color=COLORS['gforce'], linewidth=0.8, alpha=0.7)
axes[1].axhline(y=3.0, color='black', linestyle=':', linewidth=1, label='Seuil choc (3G)')
axes[1].set_ylabel('G-Force')
axes[1].set_xlabel('Temps')
axes[1].set_title('Variable Exogène — Stress Mécanique', fontweight='bold')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

print(f"\nLe modèle SARIMAX produit un MAE de {mae:.4f}°C et un RMSE de {rmse:.4f}°C.")
print("Observez comment la prédiction 'lisse' les ruptures brutales causées par les chocs.")

---
## 4. ⚠️ La Critique Industrielle — Pourquoi le Forecasting est Inadapté

<div class="alert alert-warning">
<h3>🔥 VERDICT D'INGÉNIERIE : Le Forecasting Classique est INADAPTÉ à ThermoPath</h3>
<hr>

<h4>❌ Argument 1 — L'Inertie du Modèle face aux Ruptures Brutales</h4>
<p>
Les modèles ARIMA/SARIMAX sont fondamentalement des <b>modèles de lissage</b>. Leur prédiction est une combinaison pondérée des valeurs passées et des erreurs précédentes. Ce mécanisme est <b>structurellement incapable</b> d'anticiper un saut soudain de température causé par un choc mécanique.
</p>
<p>
Concrètement : si un nid-de-poule provoque un pic de G-Force à 4.0G à l'instant <i>t</i>, la dérive thermique qui en résulte ne sera <b>détectée par le modèle qu'avec un retard de plusieurs minutes</b>, car les coefficients AR et MA sont calibrés sur des données historiques lisses. Dans le contexte de la chaîne du froid pharmaceutique (<b>vaccins à -70°C</b>), ce retard peut signifier la <b>destruction de milliers de doses</b>.
</p>

<h4>❌ Argument 2 — Coût Computationnel Prohibitif (Edge Computing)</h4>
<p>
Le déploiement cible de ThermoPath est un <b>microcontrôleur ESP32</b> ou un système embarqué SIL (Safety Integrity Level). Exécuter un modèle SARIMAX en temps réel implique :
</p>
<ul>
    <li><b>Inversion de matrices</b> à chaque prédiction — opération O(n³) en complexité.</li>
    <li><b>Stockage en mémoire</b> de l'historique complet de la série pour le recalcul des coefficients.</li>
    <li><b>Temps de calcul</b> incompatible avec une fréquence d'échantillonnage de 1 Hz (une mesure par seconde).</li>
</ul>
<p>
Un ESP32 dispose de <b>520 Ko de SRAM</b> et d'un processeur à <b>240 MHz</b>. Inverser les matrices nécessaires à un SARIMAX à chaque seconde est tout simplement <b>irréaliste</b>. À l'inverse, un Isolation Forest pré-entraîné ne nécessite qu'une <b>simple traversée d'arbres de décision</b> — opération O(log n), réalisable en microsecondes.
</p>

<h4>❌ Argument 3 — Désalignement avec le Besoin Métier</h4>
<p>
Le <b>logisticien de terrain</b> ne demande <b>jamais</b> : <i>"Quelle sera exactement la température dans 10 minutes ?"</i>. Ce qu'il veut, c'est une réponse binaire immédiate :
</p>
<p style="text-align:center; font-size:1.3em; font-weight:bold;">
🟢 "Le système est NORMAL" &nbsp;&nbsp;&nbsp; ou &nbsp;&nbsp;&nbsp; 🔴 "ALERTE — État ANORMAL détecté"
</p>
<p>
Le Forecasting fournit une <b>valeur continue</b> (ex : "-65.4°C dans 10 min") qui nécessite ensuite un seuil arbitraire pour déclencher une alerte. L'<b>Anomaly Detection</b>, en revanche, fournit nativement cette classification binaire : <b>normal (-1) vs anomalie (+1)</b>.
</p>
<p>
C'est un cas classique de <b>désalignement outil/problème</b> : le Forecasting résout un problème de <i>régression</i>, alors que notre besoin métier est un problème de <i>classification non supervisée</i>.
</p>

</div>

---
## 5. ✅ Conclusion — Le Passage à la Détection d'Anomalies

Ce notebook a démontré que l'approche classique de **prévision temporelle (Forecasting)** présente des **limitations structurelles majeures** pour le cas d'usage ThermoPath :

| Critère | Forecasting (SARIMAX) | Détection d'Anomalies (Isolation Forest) |
|---|---|---|
| **Réactivité aux chocs** | ❌ Retard dû au lissage | ✅ Détection instantanée |
| **Coût computationnel** | ❌ O(n³) — prohibitif embarqué | ✅ O(log n) — compatible ESP32 |
| **Alignement métier** | ❌ Régression continue | ✅ Classification binaire native |
| **Besoin d'étiquettes** | ⚠️ Supervisé (historique étiqueté) | ✅ Non supervisé |
| **Maintenance en production** | ❌ Recalibration fréquente | ✅ Modèle stable |

### 🎯 Décision Architecturale

La prévision pure (**Forecasting**) n'est **pas la bonne voie** pour un système embarqué de monitoring de la chaîne du froid. La solution réside dans l'**Apprentissage Non Supervisé** et plus spécifiquement dans la **Détection d'Anomalies** via **Isolation Forest**.

Cette approche :
- Apprend le **comportement normal** du système sans étiquettes.
- Fournit un **score d'anomalie** et une **décision binaire** (normal/anomalie) en temps réel.
- Est **légère en calcul**, déployable sur microcontrôleur.
- S'aligne parfaitement avec le **besoin métier** du logisticien.

<div class="alert alert-success">
<b>→ Prochaine étape :</b> Notebook 04 — Implémentation de l'Isolation Forest pour la détection d'anomalies en temps réel.
</div>